# 01 - Contexte OSM cyclable

Ce notebook extrait les objets OSM liés aux infrastructures cyclables dans les périmètres d'agglomération dissous.


In [1]:
# Paramètres
from pathlib import Path

PROJECT_ROOT = Path.cwd()
SOURCE = "vaco"
AGGLO_KEYS = ["lausanne", "berne", "geneve", "bale", "zurich"]
BOUNDARY_CRS = "EPSG:2056"
OSM_OUTPUT_CRS = "EPSG:4326"

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Exécuter ce notebook depuis la racine du dépôt projet-slow-vaud.")


In [2]:
import sys

import geopandas as gpd
import pandas as pd

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from slowvaud.osm import extract_osm_for_agglomeration
from slowvaud.paths import ensure_data_tree


## Périmètres requis

Les périmètres doivent avoir été produits par `02_agglomerations_shapes.ipynb`.


In [3]:
paths = ensure_data_tree()
boundary_dir = paths["processed"] / "agglomerations" / SOURCE
boundary_paths = {
    agglo_key: boundary_dir / f"{agglo_key}_dissolved.geojson"
    for agglo_key in AGGLO_KEYS
}
missing = [str(path) for path in boundary_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Périmètres dissous manquants. Exécuter 02_agglomerations_shapes.ipynb. " + "; ".join(missing))

boundary_checks = []
for agglo_key, path in boundary_paths.items():
    gdf = gpd.read_file(path)
    crs = str(gdf.crs)
    if crs != BOUNDARY_CRS:
        raise ValueError(f"CRS inattendu pour {path}: {crs}")
    boundary_checks.append({
        "agglomeration": agglo_key,
        "features": len(gdf),
        "crs": crs,
        "bounds_lv95": tuple(round(value, 2) for value in gdf.total_bounds),
    })

pd.DataFrame(boundary_checks)


,agglomeration,features,crs,bounds_lv95
0,lausanne,1,EPSG:2056,"(2512518.0, 1145803.0, 2551159.0, 1177123.0)"
1,berne,1,EPSG:2056,"(2581820.0, 1181565.0, 2619062.0, 1219106.0)"
2,geneve,1,EPSG:2056,"(2485424.0, 1109578.0, 2518655.0, 1155690.0)"
3,bale,1,EPSG:2056,"(2595250.0, 1246272.0, 2639000.0, 1272264.0)"
4,zurich,1,EPSG:2056,"(2664527.0, 1222925.0, 2716149.0, 1276539.9)"


## Extraction Overpass/OSMnx


In [4]:
osm_rows = []
for agglo_key, boundary_path in boundary_paths.items():
    output = extract_osm_for_agglomeration(
        agglo_key,
        boundary_path,
        boundary_crs=BOUNDARY_CRS,
    )
    gdf = gpd.read_file(output)
    crs = str(gdf.crs)
    if crs != OSM_OUTPUT_CRS:
        raise ValueError(f"CRS inattendu pour {output}: {crs}")
    osm_rows.append({
        "agglomeration": agglo_key,
        "features": len(gdf),
        "crs": crs,
        "output": str(output),
    })

pd.DataFrame(osm_rows)


,agglomeration,features,crs,output
0,lausanne,5449,EPSG:4326,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
1,berne,3773,EPSG:4326,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
2,geneve,7809,EPSG:4326,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
3,bale,7808,EPSG:4326,/Users/marc-ed/BAS-local/repos/projet-slow-vau...
4,zurich,19080,EPSG:4326,/Users/marc-ed/BAS-local/repos/projet-slow-vau...


## Classes OSM faibles


In [5]:
class_rows = []
for row in osm_rows:
    gdf = gpd.read_file(row["output"])
    counts = gdf["osm_cycle_class"].value_counts().reset_index()
    counts.columns = ["osm_cycle_class", "features"]
    counts["agglomeration"] = row["agglomeration"]
    class_rows.append(counts)

pd.concat(class_rows, ignore_index=True)[["agglomeration", "osm_cycle_class", "features"]]


,agglomeration,osm_cycle_class,features
0,lausanne,contexte_cyclable_osm,3101
1,lausanne,bande_cyclable,1510
2,lausanne,piste_cyclable_dediee,445
3,lausanne,itineraire_ou_acces_velo_designe,142
4,lausanne,voie_partagee_marquee,128
5,lausanne,piste_cyclable_associee_chaussee,123
6,berne,contexte_cyclable_osm,1685
7,berne,bande_cyclable,1392
8,berne,piste_cyclable_dediee,424
9,berne,itineraire_ou_acces_velo_designe,157
